# Sistema de Recomendação de Transporte Público — API Olho Vivo

**Parte do Membro 3: seleção e previsão** (Requisitos 04, 05 e 07 — previsão indisponível)

## Configuração e dependências dos outros membros

- `sessao`: `requests.Session` já autenticada (**Membro 1**). O token vem do `.env` e **nunca** vai para o GitHub.
- `haversine` e a lista de linhas candidatas: **Membro 2**.

Aqui ficam versões mínimas só para este trecho rodar sozinho. Na junção, apague o que o colega já tiver feito.

In [ ]:
import math

BASE_URL = "http://api.olhovivo.sptrans.com.br/v2.1"
TIMEOUT = 10          # segundos
RAIO_METROS = 500     # raio definido pelo grupo (entre 1 e 1000 m)


# --- vem do Membro 2 (versão mínima para teste) ---
def haversine(lat1, lon1, lat2, lon2):
    """Distância em metros entre dois pontos geográficos."""
    r = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))

---
## Requisito 04 — Seleção de uma opção de transporte

**Critério de escolha (justificativa):** entre as linhas candidatas, escolhemos a de **menor caminhada total**,
isto é, distância da origem até o embarque **mais** distância do desembarque até o destino. É um critério simples,
fácil de explicar e que favorece o usuário.

Regras aplicadas a cada candidata:
1. Testamos todas as duplas (embarque, desembarque) com os dois pontos **dentro do raio** e ficamos com a de menor caminhada total.
2. Embarque e desembarque precisam ser paradas diferentes.
3. Com `exigir_ordem=True`, só valem duplas em que o embarque vem **antes** do desembarque na lista de paradas.

**Teste com a API real (linha 8000):** `/Parada/BuscarParadasPorLinha` devolve as paradas **ordenadas pelo código da parada**,
e não pela ordem do trajeto (as coordenadas "pulam" de um lado para o outro). Por isso:
- `exigir_ordem` fica **desligado (`False`) por padrão**;
- as **paradas intermediárias não podem ser identificadas** com esse endpoint. O enunciado prevê isso
  ("quando sua sequência puder ser identificada"), então a lista fica vazia e o resumo informa o motivo.

In [ ]:
def melhor_dupla(paradas, origem, destino, raio, exigir_ordem):
    """
    Entre as paradas de uma linha, escolhe (embarque, desembarque) de menor caminhada total.
    Retorna (total_m, embarque, desembarque) ou None. Cada parada = (indice, dict, dist_origem, dist_destino).
    """
    validas = []
    for i, p in enumerate(paradas):
        try:
            lat, lon = float(p["py"]), float(p["px"])
        except (KeyError, TypeError, ValueError):
            continue  # parada com dados inconsistentes é ignorada
        validas.append((
            i, p,
            haversine(origem[0], origem[1], lat, lon),
            haversine(destino[0], destino[1], lat, lon),
        ))

    perto_origem = [v for v in validas if v[2] <= raio]
    perto_destino = [v for v in validas if v[3] <= raio]

    melhor = None
    for emb in perto_origem:
        for des in perto_destino:
            if emb[0] == des[0]:
                continue                      # mesma parada
            if exigir_ordem and emb[0] >= des[0]:
                continue                      # embarque depois do desembarque
            total = emb[2] + des[3]
            if melhor is None or total < melhor[0]:
                melhor = (total, emb, des)
    return melhor


def escolher_opcao(candidatas, origem, destino, raio=RAIO_METROS, exigir_ordem=False):
    """
    Escolhe a melhor linha entre as candidatas (menor caminhada total).
    Retorna (opcao, mensagem). Se nenhuma atender: (None, mensagem de erro).
    """
    if not candidatas:
        return None, "Nenhuma linha candidata foi informada."

    opcoes = []
    for c in candidatas:
        paradas = c.get("paradas") or []
        if len(paradas) < 2:
            continue

        dupla = melhor_dupla(paradas, origem, destino, raio, exigir_ordem)
        if dupla is None:
            continue

        total, emb, des = dupla
        opcoes.append({
            "codigo_linha": c.get("codigo_linha"),
            "letreiro": c.get("letreiro"),
            "sentido": c.get("sentido"),
            "embarque": emb[1],
            "desembarque": des[1],
            "dist_origem_embarque_m": round(emb[2]),
            "dist_desembarque_destino_m": round(des[3]),
            "caminhada_total_m": round(total),
            "paradas_intermediarias": paradas[emb[0] + 1:des[0]] if exigir_ordem else [],
        })

    if not opcoes:
        return None, "Nenhuma linha atende origem e destino dentro do raio informado."

    melhor = min(opcoes, key=lambda o: o["caminhada_total_m"])
    return melhor, f"{len(opcoes)} opção(ões) válida(s); escolhida a de menor caminhada."

---
## Requisito 05 — Previsão de chegada

Consulta `GET /Previsao?codigoParada=...&codigoLinha=...` e devolve o **próximo ônibus** no ponto de embarque
(mais alguns horários seguintes, quando existirem).

**Teste com a API real:** quando não há ônibus previsto, a API responde `{"hr": "16:43", "p": null}`.
Isso é tratado como "sem previsão" (e não como erro de dados).

Esta função **levanta exceções** quando algo dá errado. Quem decide como mostrar o problema ao usuário é
a seção do Requisito 07, logo abaixo.

In [ ]:
import requests


class PrevisaoIndisponivelError(Exception):
    """A previsão não pôde ser obtida (a mensagem explica o motivo)."""


def consultar_previsao(sessao, codigo_parada, codigo_linha):
    """Retorna dict com a previsão do próximo ônibus ou levanta PrevisaoIndisponivelError."""
    try:
        resp = sessao.get(
            f"{BASE_URL}/Previsao",
            params={"codigoParada": codigo_parada, "codigoLinha": codigo_linha},
            timeout=TIMEOUT,
        )
    except requests.exceptions.Timeout:
        raise PrevisaoIndisponivelError("a API demorou demais para responder.")
    except requests.exceptions.RequestException:
        raise PrevisaoIndisponivelError("falha de comunicação com a API.")

    if resp.status_code != 200:
        raise PrevisaoIndisponivelError(f"a API respondeu HTTP {resp.status_code}.")

    try:
        dados = resp.json()
    except ValueError:
        raise PrevisaoIndisponivelError("resposta da API mal formada.")

    try:
        parada = (dados or {}).get("p") or {}          # "p" pode vir como null
        linhas = parada.get("l") or []
        veiculos = [v for l in linhas for v in (l.get("vs") or [])]
    except AttributeError:
        raise PrevisaoIndisponivelError("dados inconsistentes na resposta da API.")

    if not veiculos:
        raise PrevisaoIndisponivelError("não há previsão de chegada no momento para esta linha neste ponto.")

    veiculos.sort(key=lambda v: v.get("t", "99:99"))
    proximo = veiculos[0]
    return {
        "hora_consulta": dados.get("hr"),
        "proximo_horario": proximo.get("t"),
        "prefixo_veiculo": proximo.get("p"),
        "acessivel": proximo.get("a"),
        "outros_horarios": [v.get("t") for v in veiculos[1:4]],
    }

---
## Requisito 07 — Tratamento de erros (parte do Membro 3)

Duas situações cobertas aqui:
- **Nenhuma linha atende** os critérios → `escolher_opcao` devolve `None` e uma mensagem.
- **Previsão indisponível** → o erro de `consultar_previsao` vira um **aviso** para o usuário, sem derrubar o programa
  (a viagem recomendada continua sendo exibida, só sem o horário).

In [ ]:
def obter_previsao_segura(sessao, embarque, codigo_linha):
    """Retorna (previsao, aviso). Nunca levanta exceção: se falhar, previsao=None e aviso explica."""
    try:
        previsao = consultar_previsao(sessao, embarque["cp"], codigo_linha)
        return previsao, None
    except PrevisaoIndisponivelError as e:
        return None, f"⚠️ Previsão indisponível: {e}"
    except KeyError:
        return None, "⚠️ Previsão indisponível: ponto de embarque sem código."


def recomendar(sessao, candidatas, origem, destino, raio=RAIO_METROS, exigir_ordem=False):
    """Junta Requisitos 04, 05 e 07. Retorna dict com opcao, previsao e lista de avisos."""
    avisos = []

    opcao, msg = escolher_opcao(candidatas, origem, destino, raio, exigir_ordem)
    if opcao is None:
        return {"opcao": None, "previsao": None, "avisos": [f"⚠️ {msg}"]}

    previsao, aviso = obter_previsao_segura(sessao, opcao["embarque"], opcao["codigo_linha"])
    if aviso:
        avisos.append(aviso)

    return {"opcao": opcao, "previsao": previsao, "avisos": avisos}

---
## Demonstração (dados fictícios, sem chamar a API)

Para testar sem token, usamos uma sessão falsa que imita a resposta da API.
Na demonstração real, troque `SessaoFalsa` pela `sessao` autenticada do Membro 1 e `candidatas` pela lista do Membro 2.

In [ ]:
class _RespostaFalsa:
    def __init__(self, status, corpo):
        self.status_code, self._corpo = status, corpo
    def json(self):
        return self._corpo


class SessaoFalsa:
    """Imita requests.Session para testar a previsão sem a API."""
    def __init__(self, status=200, corpo=None):
        self.status, self.corpo = status, corpo
    def get(self, url, params=None, timeout=None):
        return _RespostaFalsa(self.status, self.corpo)


origem, destino = (-23.5614, -46.6559), (-23.5505, -46.6333)
candidatas = [
    {"codigo_linha": 1, "letreiro": "8000-10", "sentido": 1, "paradas": [
        {"cp": 10, "np": "Paulista A", "py": -23.5612, "px": -46.6557},
        {"cp": 11, "np": "Meio",       "py": -23.5560, "px": -46.6450},
        {"cp": 12, "np": "Centro",     "py": -23.5507, "px": -46.6335}]},
    {"codigo_linha": 2, "letreiro": "7000-10", "sentido": 1, "paradas": [   # fica longe: deve ser descartada
        {"cp": 20, "np": "Longe 1", "py": -23.6000, "px": -46.7000},
        {"cp": 21, "np": "Longe 2", "py": -23.6100, "px": -46.7100}]},
]

resposta_api = {"hr": "14:32", "p": {"cp": 10, "l": [{"vs": [
    {"p": "11001", "t": "14:38", "a": True},
    {"p": "11002", "t": "14:50", "a": False}]}]}}

resposta_sem_previsao = {"hr": "16:43", "p": None}   # formato real quando não há ônibus previsto

In [ ]:
# Caso normal: previsão disponível (paradas na ordem em que a API devolve, sem exigir trajeto)
r = recomendar(SessaoFalsa(200, resposta_api), candidatas, origem, destino)
o = r["opcao"]
print(f"Linha {o['letreiro']} (sentido {o['sentido']})")
print(f"Embarque:    {o['embarque']['np']} a {o['dist_origem_embarque_m']} m da origem")
print(f"Desembarque: {o['desembarque']['np']} a {o['dist_desembarque_destino_m']} m do destino")
print("Caminhada total:", o["caminhada_total_m"], "m")
print("Paradas entre os dois:", [p["np"] for p in o["paradas_intermediarias"]] or "não identificáveis (a API não devolve a ordem do trajeto)")
print("Próximo ônibus:", r["previsao"]["proximo_horario"], "| depois:", r["previsao"]["outros_horarios"])
print("Avisos:", r["avisos"])

In [ ]:
# Com exigir_ordem=True (só se a ordem das paradas fosse a do trajeto): aparecem as paradas intermediárias
r = recomendar(SessaoFalsa(200, resposta_api), candidatas, origem, destino, exigir_ordem=True)
print("Paradas entre os dois:", [p["np"] for p in r["opcao"]["paradas_intermediarias"]])

# Linha com paradas em ordem invertida: exigir_ordem=True descarta, exigir_ordem=False aceita
invertida = [{"codigo_linha": 3, "letreiro": "9000-10", "sentido": 1,
              "paradas": list(reversed(candidatas[0]["paradas"]))}]
print("Ordem exigida:   ", recomendar(SessaoFalsa(200, resposta_api), invertida, origem, destino, exigir_ordem=True)["avisos"])
print("Ordem não exigida:", recomendar(SessaoFalsa(200, resposta_api), invertida, origem, destino)["opcao"]["letreiro"])

In [ ]:
# Erro 1 (R07): previsão indisponível, no formato real da API ("p": null) -> a recomendação continua, com aviso
r = recomendar(SessaoFalsa(200, resposta_sem_previsao), candidatas, origem, destino)
print("Linha recomendada:", r["opcao"]["letreiro"])
print("Previsão:", r["previsao"])
print("Avisos:", r["avisos"])

# Erro 2 (R07): API com erro HTTP
r = recomendar(SessaoFalsa(500, None), candidatas, origem, destino)
print("Avisos:", r["avisos"])

# Erro 3 (R07): nenhuma linha atende ao raio
r = recomendar(SessaoFalsa(200, resposta_api), candidatas, origem, destino, raio=5)
print("Opção:", r["opcao"], "| Avisos:", r["avisos"])